# HRL Imperviousness — Time Series Reconstruction
## Propagating Modern Accuracy Backwards in Time

---

The **Copernicus Land Monitoring Service (CLMS)** provides geographical information on land cover and its changes, land use, vegetation state, water cycle and earth surface energy variables to a broad range of users in Europe and across the world for various domains and applications. CLMS is jointly implemented by the **European Environment Agency (EEA)** and the European Commission’s Directorate-General **Joint Research Centre (JRC)**. 

The **High-Resolution Layer Imperviousness (HRL Imperviousness)** is part of the pan-European CLMS portfolio that currently covers the EEA38+UK countries. It consists of two products: Imperviousness and Impervious Built-Up, along with their derived and supporting layers. All layers are derived from high-resolution optical satellite image time series (Sentinel-2) via automatic image processing methods and provide dedicated information on **Impervious and Built-Up** surfaces and detected dynamics between two reference years.
Further information on the **Imperviousness Density (IMD)** can be found in the: [PUM](https://land.copernicus.eu/en/technical-library/product-user-manual-high-resolution-layer-imperviousness-2024) or [ATBD](https://land.copernicus.eu/en/technical-library/algorithm-theoretical-basis-document-high-resolution-layer-imperviousness-2024).

**HRL Imperviousness** incl. its main dataset, **Imperviousness Density (IMD)**, is updated at a 3-year frequency, incorporating continuously improved algorithms and refined calibration. While recent editions leverage modern high-revisit constellations, pre-Sentinel products relied on Landsat data characterized by a native 30 m resolution and sparser observations, occasionally resulting in data gaps (e.g., in the 2006 reference year) and resolution shifts. Combining modern sensor capabilities with ongoing algorithm enhancements creates an opportunity:

> *Can we use the accuracy of the most recent layer to improve our representation of historical conditions?*

The answer is **yes** — but the approach matters. 

This notebook demonstrates why the most intuitive approach—directly subtracting detected change—introduces physically impossible values in the time series. While a full, centralized reprocessing using unified algorithms and common calibration across all historical epochs would be the gold standard, end users require practical post-processing solutions today. To address this, we introduce the **Binary Mask Substitution** method as a robust alternative. This approach relies on the conservative assumption that impervious surfaces identified in current High-Resolution Layers were already present in earlier reference years, effectively eliminating legacy data gaps across the 2006–2015 harmonized series without introducing negative density artifacts.

### What you will learn

| Section | Topic |
|---------|-------|
| **Intro** | Data products, site selection, baseline visualisation |
| **Section 1** | Why backward subtraction fails and produces negative imperviousness values |
| **Section 2** | The binary mask substitution that avoids these problems |
| **Section 3** | Interactive comparison and exploration |
| **Section 4** | Limitations and recommendations |

### Key HRL Imperviousness layers

| Layer | Description | Values |
|---------|------------|--------|
| **IMD** | Imperviousness Density — sealed surface fraction per pixel | 0–100 % |
| **IMDC** | Imperviousness Density Change — detected change between epochs | encoded bytes |

**IMDC encoding:** Raw values are unsigned bytes (0–255) using an offset of 100:
- `100` = no change (0 %)
- `113` = +13 % increase
- `87` = −13 % decrease
- `201` = technical no-change class
- `255` = nodata

In [ ]:
import numpy as np

from helpers.raster_utils import normalize_change, match_to_grid, bbox_to_4326
from helpers.data_loading import load_sites, load_site_layers
from helpers.analysis import summarize_invalid, print_invalid_summary
from helpers.plotting import (
    plot_site_overview, plot_status_map, plot_subtraction_maps, plot_subtraction_histograms,
    plot_method_comparison, plot_invalid_pixel_counts, plot_year_distribution, plot_interactive_comparison,
)
from helpers.ui import make_dropdown

print('Environment ready.')


In [ ]:
# ── Timeline ─────────────────────────────────────────────────────────────────────────────
YEARS        = ['24', '21', '18', '15', '12', '09', '06']   # newest to oldest
PAIRS        = ['2124', '1821', '1518', '1215', '0912', '0609']
NON_BASELINE = YEARS[1:]

# ── Site definitions ───────────────────────────────────────────────────────────────
SITES = load_sites('data/site_definition.yaml')
print('Available sites:', list(SITES.keys()))


---
## Data and Site Selection

This notebook uses two layers of the **HRL Imperviousness**: the Imperviousness Density (IMD) status layers and the Imperviousness Density Change (IMDC) layers.

### IMD — Imperviousness Density Status Layers

Seven snapshots of sealed-surface density, one per epoch:

| Year | Role in this notebook |
|------|----------------------|
| 2024 | **Baseline** — most recent, highest quality |
| 2021–2006 | Historical reference layers |

Each pixel value represents the percentage of impervious surface (0–100 %).

### IMDC — Imperviousness Density Change Layers

Six paired change layers, each encoding detected real change between two consecutive epochs.
After normalising (subtracting the offset 100), values represent change in percentage points:
positive = new sealed surface, negative = de-sealing.
Throughout this notebook, the term **"change mask"** refers to this IMDC layer after
binarisation (change / no-change).

### Demo sites

The two demonstration sites in this notebook are shown at different resolutions, on purpose:

- **North-Italy** uses matched 100 m status and change layers — the simplest case, where no
  resampling is needed, and used here to introduce the method.
- **Innsbruck** uses 10 m status with 20 m change layers, to demonstrate the cross-resolution
  case: the 10 m IMD status layers (available from 2018 onwards) must be reconciled with the
  coarser, 20 m change layers and status layers from earlier epochs. This mirrors the
  actual resolution transition present in the product time series.

### Cross-resolution note

For the 10 m study area, the IMDC layer has a **native resolution of 20 m**. This mismatch is handled as follows:

1. Derive the binary change mask from the IMDC layer (by binarisation: change / no-change)
2. Resample the **mask only** to 10 m using nearest-neighbour
3. Apply the 10 m mask to the 10 m status layer

Resampling the mask (not the raw values) is important: nearest-neighbour of a binary array yields exact copies of the original 0/1 values with no fractional artefacts.

In [ ]:
region_selector = make_dropdown(list(SITES.keys()), 'Region:')


In [ ]:
SELECTED_SITE = region_selector.value
cfg        = SITES[SELECTED_SITE]
STATUS_RES = cfg['status_res']
CHANGE_RES = cfg['change_res']
XMIN, XMAX = cfg['xmin'], cfg['xmax']
YMIN, YMAX = cfg['ymin'], cfg['ymax']
ZOOM       = cfg['zoom']

CLAT, CLON, BOUNDS_4326 = bbox_to_4326(XMIN, YMIN, XMAX, YMAX, cfg['target_srid'])

# True when the change (IMDC) layer has a coarser resolution than the status
# (IMD) layer; triggers the mask-resampling step used throughout the notebook.
CROSS_RES = STATUS_RES != CHANGE_RES

print(f'Study area   : {SELECTED_SITE}')
print(f'Status res   : {STATUS_RES} m')
print(f'Change res   : {CHANGE_RES} m')
print(f'Extent       : {XMIN}–{XMAX} E,  {YMIN}–{YMAX} N  (EPSG:{cfg["target_srid"]})')
print(f'Center       : {CLAT:.5f} N, {CLON:.5f} E  (EPSG:4326)')


In [ ]:
# Interactive map showing the study area location in Europe
plot_site_overview(CLAT, CLON, ZOOM, BOUNDS_4326, SELECTED_SITE)


In [ ]:
site = load_site_layers(SELECTED_SITE, cfg, YEARS, PAIRS)
STATUS, CHANGE, REF_DS = site['status'], site['change'], site['ref_ds']


In [ ]:
plot_status_map(
    STATUS['24'][0],
    f'IMD 2024 — {SELECTED_SITE}  (Baseline, {STATUS_RES} m resolution)',
)


---
## Section 1: Backward Subtraction Method

### The intuitive approach

The most natural way to reconstruct the 2021 layer from the 2024 baseline is:

$$\text{IMD}_r(2021) = \text{IMD}_r(2024) - \text{IMDC}(2021 \to 2024)$$

and continue backwards for each epoch. This is straightforward to implement and appears logically sound. However, it contains a critical flaw.

### Why it fails: three overlapping problems

**1. Time-bound validity of change estimates**
Each IMDC layer is valid only between its two specific observation years. The absolute magnitude of change between 2018 and 2021 cannot be safely applied to a 2024 baseline that was produced with a completely different algorithm version.

**2. Technical corrections masquerade as real changes**
When a newer algorithm corrects a systematic error from a previous run (e.g. an overestimated 25 % pixel is corrected to 0 %), the change layer records a −25 % change. Subtracting this backwards would produce +25 % at the historical position — but that 25 % never actually existed physically.

**3. Cumulation of errors**
With each backward step, errors accumulate. A pixel at 0 % in 2024 that has +13 % change between 2018 and 2021 will be reconstructed as −13 % for 2018. **Negative imperviousness is physically impossible.**

### A worked example

Consider a single pixel tracked through the time series:

**Table 1 — Status values**

| Year | IMD (reported) | IMD reconstructed (subtraction) |
|------|---------------|--------------------------------|
| 2024 | 0 % | 0 % (= baseline) |
| 2021 | 25 % | **0 %** (should be 25 %) |
| 2018 | 0 % | **−13 %** (impossible!) |

**Table 2 — Detected changes**

| Period | Real change (IMDC) | Technical change |
|--------|-------------------|------------------|
| 2021→2024 | 0 % | +25 % (algorithm correction) |
| 2018→2021 | +13 % | +12 % |

Step-by-step reconstruction:
- `IMD_r(2024) = 0 %`
- `IMD_r(2021) = 0 − 0 = 0 %`  ← wrong (real value is 25 %)
- `IMD_r(2018) = 0 − 13 = −13 %`  ← **impossible**

The code below applies this method and visualises where values fall outside the valid 0–100 % range.

In [ ]:
print('Applying backward subtraction ...')

sub_results  = {}
sub_baseline = STATUS['24'][0].copy()

for pair, prev_year, curr_year in zip(PAIRS, YEARS[1:], YEARS[:-1]):
    raw_change, ch_ds = CHANGE[pair]
    norm_change = normalize_change(raw_change)

    # Resample normalised change values to status grid when resolutions differ
    if CROSS_RES:
        norm_change = match_to_grid(norm_change, ch_ds, REF_DS)

    result = sub_baseline - norm_change
    result[np.isnan(STATUS[prev_year][0])] = np.nan  # preserve nodata

    sub_results[prev_year] = result
    sub_baseline = result.copy()

INVALID = summarize_invalid(sub_results, NON_BASELINE)
print_invalid_summary(INVALID, NON_BASELINE)


In [ ]:
plot_subtraction_maps(sub_results, INVALID, NON_BASELINE, SELECTED_SITE)


In [ ]:
plot_subtraction_histograms(sub_results, NON_BASELINE)


Let's look at the time series: how does the number of invalid pixels evolve as we reconstruct further back in time?

In [ ]:
plot_invalid_pixel_counts(INVALID, NON_BASELINE, SELECTED_SITE)

### Discussion

The histograms and maps above show a clear trend: **the further back in time, the more invalid pixels accumulate**. This is the mathematical consequence of the cumulation issue — every backward step can push additional pixels outside the valid 0–100 % range.

The fundamental problem is that simple subtraction **treats the IMDC layer as an absolute quantity** — which it is not. The change layer tells us that something changed, and by how much *in relative terms between those two specific years*. It does not tell us what the "true" historical value should be, especially when algorithm updates have changed the baseline.

The next section presents a method that avoids this entirely.

---
## Section 2: Binary Mask Substitution

### The core idea: use change as a switch, not a value

Instead of asking *how much* did the surface change, we ask *did it change at all?*

For each pixel and each backward step:

| IMDC says... | Physical interpretation | Action |
|--------------|------------------------|--------|
| **Real change** (IMDC ≠ 0) | The surface physically changed between these two years | Adopt the **original historical status** value |
| **No real change** (IMDC = 0) | The surface stayed the same | Keep the **modern baseline** value (it is more accurate) |

By never subtracting a quantity, the output is always a copy of a real, valid status value. **Negative values are impossible by construction.**

### Handling the cross-resolution case (Innsbruck: 20 m IMDC + 10 m IMD)

When the change layer is coarser than the status layer:

1. Compute the binary mask (change / no-change) at the **native 20 m** IMDC resolution
2. Resample the **mask** to **10 m** using nearest-neighbour
3. Apply to the 10 m status and baseline layers

Each 10 m pixel simply inherits the change flag of its enclosing 20 m cell. No interpolation of change values occurs.

### Pseudocode

```python
running_baseline = IMD_2024  # start with the most accurate layer

for step in [2021, 2018, 2015, 2012, 2009, 2006]:
    # 1. Derive binary mask at native IMDC resolution
    norm_change = normalize(IMDC[step, next_step])
    change_mask = (norm_change != 0)          # True where real change occurred

    # 2. Resample mask to status grid (nearest-neighbour)
    change_mask = resample_NN(change_mask, target_resolution)

    # 3. Binary mask substitution
    result = running_baseline.copy()
    result[change_mask] = IMD_historical[step][change_mask]

    running_baseline = result  # carry forward for next step
```

Note: the exact IMDC value is **never used**. Only the presence or absence of change matters. Therefore, the masking can also be done using the classified change layer (IMCC).

In [ ]:
print('Applying binary mask substitution ...')

ind_results  = {}
ind_baseline = STATUS['24'][0].copy()

for pair, prev_year, curr_year in zip(PAIRS, YEARS[1:], YEARS[:-1]):
    raw_change, ch_ds = CHANGE[pair]
    hist_status       = STATUS[prev_year][0]

    # Build binary change mask at native IMDC resolution
    norm_change = normalize_change(raw_change)
    mask_native = (norm_change != 0).astype(np.float32)  # float needed for GDAL write

    # Resample mask to status grid (NN preserves binary values exactly)
    if CROSS_RES:
        mask_resampled = match_to_grid(mask_native, ch_ds, REF_DS) > 0.5
    else:
        mask_resampled = mask_native.astype(bool)

    # binary mask substitution: where change occurred, adopt historical value
    result       = ind_baseline.copy()
    valid_change = mask_resampled & ~np.isnan(hist_status)
    result[valid_change] = hist_status[valid_change]

    ind_results[prev_year] = result
    ind_baseline = result.copy()
    print(f'  20{prev_year}: {int(np.sum(valid_change)):,} pixels updated from historical layer')

# Verify: binary mask substitution should produce zero invalid pixels
IND_INVALID = summarize_invalid(ind_results, NON_BASELINE)
if all(n == 0 for n, _, _ in IND_INVALID.values()):
    print('Verification: all binary mask substitution results are within valid 0-100 % range.')
else:
    print('WARNING: unexpected invalid pixels:', {y: v[0] for y, v in IND_INVALID.items() if v[0]})


In [ ]:
year_selector = make_dropdown(sorted(NON_BASELINE), 'Choose year:')


In [ ]:
year = year_selector.value
plot_method_comparison(STATUS[year][0], sub_results[year], ind_results[year], INVALID[year], year)


In [ ]:
plot_year_distribution(STATUS, sub_results, ind_results, year)

---
## Section 3: Interactive Exploration

Use the year selector from Section 2 above to pick a year of interest. The map below lets you
toggle between the original status layer, the subtraction result, and the binary mask
substitution result for that year.

Re-run the cell after changing the year.

In [ ]:
year = year_selector.value
plot_interactive_comparison(
    CLAT, CLON, ZOOM, BOUNDS_4326,
    STATUS[year][0], sub_results[year], ind_results[year], year,
)


---
## Section 4: Limitations

The binary mask substitution is effective for improving the visual consistency of a time series. Before using it, however, users must understand its key limitations.

---

### 1. Change detection errors propagate backwards

The reconstruction inherits any errors present in the IMDC layers:

- **Change omissions**: If a real change occurred but was not detected, the modern baseline is propagated backward. A building constructed in 2009 that was missed by the change detector will appear in the 2006 reconstruction as a "ghost" impervious surface — even though it did not physically exist then.

- **Change commissions**: If a spurious change is detected where none occurred, the reconstruction incorrectly adopts the historical status value, potentially introducing visual artefacts.

- **Baseline errors**: Any error already present in IMD 2024 is carried unchanged into every earlier year where no real change was detected.

The quality of the reconstruction is bounded by the quality of the change detection.

---

### 2. This method is for visual quality — not for change quantification

The reconstructed layers are designed to provide a **spatially consistent and visually coherent** time series. They are explicitly **not** intended for quantitative change analysis.

> **If you need to measure how much imperviousness has changed, always use the original IMDC layers directly.**

| Question | Correct tool |
|----------|--------------
| How does the map look for 2009? | Binary mask substitution IMD layer |
| How many hectares of new impervious surface appeared 2009–2012? | IMDC layer (aggregate the continuous density change values) |
| What is the total sealed surface area in 2015? | Original IMD 2015 layer |

Do **not** count pixels in a classified change product. Do **not** subtract status layers from each other. Always aggregate the continuous density change values from the IMDC layer.

---

### 3. Uncertainty increases with temporal distance from the baseline

The 2021 reconstruction carries uncertainty from one change epoch. The 2006 reconstruction carries accumulated uncertainty from all six. Treat the earliest reconstructed layers with more caution.

---

### 4. Spatial precision of changed areas is limited by the change layer resolution

For sites with a coarser IMDC layer (e.g. 20 m IMDC used with 10 m IMD status), the **boundaries** of changed areas in the reconstruction are limited to 20 m precision. The 10 m IMD values within those areas are correct, but the spatial sharpness of change edges is determined by the 20 m mask.

---

### 5. Why not a full reprocessing?

A full reprocessing with the current algorithm and common calibration across all epochs would in principle be the most correct solution. Binary mask substitution is a lower-cost alternative to that; it assumes a pixel's current status is representative of its historical state wherever no real change was detected. Its main benefit is on the already harmonised 2006–2015 layers, where it removes data gaps and increases the resolution to 10m from that period without a new processing run.

---

### Summary

| Use case | Use binary mask substitution? |
|----------|------------------------------|
| Visualise historical imperviousness maps | **Yes** |
| Create a consistent time series for cartographic products | **Yes** |
| Quantify annual imperviousness change | **No** — use IMDC (aggregate density change) |
| Compute total impervious area per year | **No** — use original IMD |
| Detect individual land-cover change events | **No** — use IMDC |
| Pixel-by-pixel "historical change history" | With caution — errors propagate |